# Session 4 Part B: Transfer Learning with ResNet50

## Fine-Tuning Pre-Trained Models for Earth Observation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anefelo/eo-ai-ml-training/blob/main/day2/notebooks/session4_transfer_learning_STUDENT.ipynb)

**EO AI/ML Training · Day 2, Session 4 (Part B)** | Instructor: Dimitrios Kasampalis (Anefelo)

**Duration:** 60 minutes | **Difficulty:** Intermediate-Advanced  
**Dataset:** EuroSAT RGB (same splits as Part A)

## 🧰 What you'll need

- Google Colab with a **GPU runtime** (ResNet50 at 224×224 is too slow on CPU)
- TensorFlow 2.16+ (Keras 3), `tensorflow-datasets`; ImageNet weights (~95 MB) download automatically
- Optional: `test_predictions.csv` from Part A, to compare against your from-scratch CNN

---

## 🎯 Objectives

By the end of this notebook, you will:

1. ✅ Understand transfer learning concepts
2. ✅ Load pre-trained ResNet50 (ImageNet weights)
3. ✅ Apply the preprocessing the pre-trained weights expect
4. ✅ Fine-tune the model on EuroSAT
5. ✅ Compare transfer learning vs from-scratch CNN
6. ✅ Reach roughly 95%+ test accuracy (typically above the Part A CNN)

---

## 📋 What is Transfer Learning?

**Concept:** Use knowledge from one task to improve performance on another

**How it works:**
1. Take a model pre-trained on large dataset (e.g., ImageNet: 1.2M images, 1000 classes)
2. Remove final classification layer
3. Add new layers for your specific task
4. Fine-tune on your dataset (smaller, specialized)

**Benefits:**
- ✅ **Less data needed:** 1000s instead of millions
- ✅ **Faster training:** Start from good features
- ✅ **Better accuracy:** Leverage learned representations
- ✅ **Prevents overfitting:** Pre-trained weights are robust

**ImageNet → EuroSAT:**
- ImageNet learned: edges, textures, shapes, objects
- We adapt: Apply these features to satellite imagery

---

## 🏗️ Notebook Structure

| Step | Activity | Duration |
|------|----------|----------|
| **1** | Setup & Load Pre-trained Model | 10 min |
| **2** | Load Pre-trained ResNet50 | 5 min |
| **3** | Feature Extraction (Freeze Base) | 15 min |
| **4** | Fine-Tuning (Unfreeze Layers) | 20 min |
| **5** | Comparison & Analysis | 10 min |

---

Let's leverage pre-trained models! 🚀

---

# Step 1: Setup & Load Pre-trained Model (10 minutes)

We'll use the same environment and dataset from Part A.


In [ ]:
# Imports (same as Part A)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print(f"✓ TensorFlow version: {tf.__version__}")
print(f"✓ Keras version: {keras.__version__}")

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✓ GPU available: {len(gpus)} device(s)")
else:
    print("\n⚠️  No GPU - training will be slower")

# Set seeds (Python, NumPy, TensorFlow)
keras.utils.set_random_seed(42)

### Load Dataset (Same as Part A)

If you completed Part A in the same session, the dataset is already downloaded. Otherwise, we'll re-download.


In [ ]:
# Load EuroSAT (same source and same 70/15/15 split as Part A)
print("Loading EuroSAT dataset...")

# EuroSAT RGB: 27,000 Sentinel-2 chips (64x64 px, 10 classes), MIT licence.
# Source: the authors' Zenodo record. (The old TensorFlow Datasets download host is offline,
# so we fetch the zip directly and build the same 70/15/15 split ourselves.)
import pathlib

DATA_URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"
archive = pathlib.Path(keras.utils.get_file("EuroSAT_RGB.zip", DATA_URL, extract=True))
search_root = archive if archive.is_dir() else archive.parent
data_dir = next(p.parent for p in sorted(search_root.rglob("AnnualCrop")) if p.is_dir())

class_names = sorted(d.name for d in data_dir.iterdir() if d.is_dir())
num_classes = len(class_names)
paths, labels = [], []
for idx, name in enumerate(class_names):
    files = sorted(str(f) for f in (data_dir / name).glob("*.jpg"))
    paths += files
    labels += [idx] * len(files)

# Fixed shuffle, then 70% train / 15% validation / 15% test (reproducible)
order = np.random.default_rng(42).permutation(len(paths))
paths, labels = np.array(paths)[order], np.array(labels, dtype="int64")[order]
n_train, n_val = int(0.70 * len(paths)), int(0.15 * len(paths))

def make_ds(p, y):
    """(file path, label) -> (uint8 image 64x64x3, label)"""
    def load(f, label):
        img = tf.io.decode_jpeg(tf.io.read_file(f), channels=3)
        return tf.ensure_shape(img, (64, 64, 3)), label
    return tf.data.Dataset.from_tensor_slices((p, y)).map(load, num_parallel_calls=tf.data.AUTOTUNE)

ds_train = make_ds(paths[:n_train], labels[:n_train])
ds_val = make_ds(paths[n_train:n_train + n_val], labels[n_train:n_train + n_val])
ds_test = make_ds(paths[n_train + n_val:], labels[n_train + n_val:])

print(f"\n✓ Dataset loaded")
print(f"  Classes: {num_classes}")
print(f"  Train: {ds_train.cardinality().numpy()} images")
print(f"  Val: {ds_val.cardinality().numpy()} images")
print(f"  Test: {ds_test.cardinality().numpy()} images")

### Preprocessing & Batching

In [ ]:
# Preprocessing for ResNet50
# Keras' ImageNet ResNet50 weights expect images in 0-255 that are then passed through
# keras.applications.resnet50.preprocess_input (RGB→BGR + ImageNet mean subtraction).
# Feeding [0, 1] images instead silently degrades the pre-trained features.
from tensorflow.keras.applications.resnet50 import preprocess_input

IMG_SIZE = 224   # ResNet50's native size; 128 or 160 trains ~2-3x faster with a small accuracy cost

def resize(image, label):
    image = tf.image.resize(tf.cast(image, tf.float32), [IMG_SIZE, IMG_SIZE])   # 64x64 → IMG_SIZE, values stay 0-255
    return image, label

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomBrightness(0.1),            # default value_range=(0, 255) matches these images
], name='augmentation')

def augment(image, label):
    return tf.clip_by_value(data_augmentation(image, training=True), 0.0, 255.0), label

def to_resnet(image, label):
    return preprocess_input(image), label

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
ds_train = (ds_train.map(resize, num_parallel_calls=AUTOTUNE).shuffle(1000)
            .map(augment, num_parallel_calls=AUTOTUNE).map(to_resnet, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))
ds_val = ds_val.map(resize, num_parallel_calls=AUTOTUNE).map(to_resnet).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test = ds_test.map(resize, num_parallel_calls=AUTOTUNE).map(to_resnet).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("\n✓ Data prepared for ResNet50")
print(f"  Image size: {IMG_SIZE}×{IMG_SIZE}, ImageNet preprocessing applied")
print(f"  Batch size: {BATCH_SIZE}")

---

# Step 2: Load Pre-trained ResNet50

ResNet50 architecture:
- 50 layers deep
- Skip connections (residual blocks)
- ~25 million parameters
- Pre-trained on ImageNet (1.2M images, 1000 classes)

We'll load it **without the top classification layer** (include_top=False).


In [ ]:
# Load pre-trained ResNet50
print("Loading pre-trained ResNet50...")

base_model = ResNet50(
    include_top=False,           # Exclude ImageNet classifier
    weights='imagenet',           # Use ImageNet pre-trained weights
    input_shape=(IMG_SIZE, IMG_SIZE, 3),   # EuroSAT RGB, resized
    pooling='avg'                # Global average pooling
)

print(f"\n✓ ResNet50 loaded")
print(f"  Total parameters: {base_model.count_params():,}")
print(f"  Trainable: {base_model.trainable}")
print(f"  Output shape: {base_model.output_shape}")

### Freeze Base Model

For **feature extraction**, we freeze all ResNet50 layers initially. This means:
- Pre-trained weights don't change
- Only train the new classification head
- Much faster training
- Prevents destroying good features


In [ ]:
# Freeze the base model
base_model.trainable = False

print("✓ Base model frozen")
print(f"  Trainable parameters: {sum(int(np.prod(w.shape)) for w in base_model.trainable_weights):,}")
print(f"  Non-trainable parameters: {sum(int(np.prod(w.shape)) for w in base_model.non_trainable_weights):,}")

---

# Step 3: Feature Extraction - Train Classification Head (15 minutes)

Now we add our custom classification layers on top of frozen ResNet50.

**Architecture:**
```
Input (IMG_SIZE×IMG_SIZE×3, default 224)
    ↓
ResNet50 Base (frozen) → 2048 features
    ↓
Dense(512, ReLU) + Dropout
    ↓
Dense(10, Softmax)
```

In [ ]:
# Build complete model with custom head
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# ResNet50 base. training=False keeps BatchNorm layers in inference mode,
# which matters later when we unfreeze part of the base for fine-tuning.
x = base_model(inputs, training=False)

# Custom classification head
x = layers.Dense(512, activation='relu', name='fc1')(x)
x = layers.Dropout(0.5, name='dropout1')(x)
outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

# Create model
model_feature_extraction = keras.Model(inputs, outputs, name='ResNet50_FeatureExtraction')

print("✓ Model with custom head created")
model_feature_extraction.summary()

In [ ]:
# Compile model
model_feature_extraction.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled for feature extraction")
head_params = sum(int(np.prod(w.shape)) for w in model_feature_extraction.trainable_weights)
print(f"  Only training the classification head ({head_params:,} parameters)")

### Train Classification Head

In [ ]:
# Callbacks
callbacks_fe = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('resnet50_feature_extraction.keras', monitor='val_accuracy', 
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

# Train
print("Training classification head (feature extraction)...")
print("=" * 70)

t0 = time.time()
history_fe = model_feature_extraction.fit(
    ds_train,
    validation_data=ds_val,
    epochs=10,  # the head converges quickly
    callbacks=callbacks_fe,
    verbose=1
)

print("=" * 70)
time_fe = time.time() - t0
print(f"\n✓ Feature extraction training complete ({time_fe/60:.1f} min)")

### Evaluate Feature Extraction Model

In [ ]:
# Evaluate on test set
test_loss_fe, test_acc_fe = model_feature_extraction.evaluate(ds_test, verbose=0)

print(f"\n📊 Feature Extraction Results:")
print(f"   Test Accuracy: {test_acc_fe*100:.2f}%")
print(f"   Test Loss: {test_loss_fe:.4f}")

if test_acc_fe > 0.92:
    print("\n🎉 Excellent — the frozen ImageNet features already work well on EuroSAT")

---

# Step 4: Fine-Tuning - Unfreeze Some Layers (15 minutes)

Now we'll **fine-tune** by unfreezing the last few layers of ResNet50.

**Strategy:**
1. Unfreeze the last 20 layers of ResNet50 (the cell below prints the total layer count)
2. Use very low learning rate (1e-5)
3. Allow model to adapt to satellite imagery

**Why this works:**
- Early layers (edges, textures) are general → keep frozen
- Later layers (objects, semantics) need adaptation → unfreeze

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# But keep early layers frozen
print(f"Total layers in ResNet50: {len(base_model.layers)}")

# Freeze all layers except last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

print(f"\n✓ Fine-tuning configuration:")
print(f"  Frozen layers: {len([l for l in base_model.layers if not l.trainable])}")
print(f"  Trainable layers: {len([l for l in base_model.layers if l.trainable])}")

# Count trainable parameters (base + head)
trainable_params = sum(int(np.prod(w.shape)) for w in model_feature_extraction.trainable_weights)
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# Recompile with lower learning rate
model_feature_extraction.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # Much lower!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Model recompiled for fine-tuning")
print("  Learning rate: 1e-5 (100x smaller than in feature extraction)")

### Fine-Tune the Model

In [ ]:
# Callbacks for fine-tuning
callbacks_ft = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint('resnet50_finetuned.keras', monitor='val_accuracy', 
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-8, verbose=1)
]

# Continue training (fine-tuning)
print("Fine-tuning model...")
print("=" * 70)

t0 = time.time()
history_ft = model_feature_extraction.fit(
    ds_train,
    validation_data=ds_val,
    epochs=10,  # raise to 20-30 if you have time; early stopping guards against overfitting
    callbacks=callbacks_ft,
    verbose=1
)

print("=" * 70)
time_ft = time.time() - t0
print(f"\n✓ Fine-tuning complete ({time_ft/60:.1f} min)")

### Evaluate Fine-Tuned Model

In [ ]:
# Evaluate on test set
test_loss_ft, test_acc_ft = model_feature_extraction.evaluate(ds_test, verbose=0)

print(f"\n📊 Fine-Tuned Model Results:")
print(f"   Test Accuracy: {test_acc_ft*100:.2f}%")
print(f"   Test Loss: {test_loss_ft:.4f}")

# Compare with feature extraction
improvement = (test_acc_ft - test_acc_fe) * 100
print(f"\n   Change vs feature extraction: {improvement:+.2f} percentage points")

if test_acc_ft > 0.94:
    print("\n🎉 Outstanding! >94% accuracy achieved!")
    print("   Transfer learning + fine-tuning is very effective")

---

# Step 5: Comparison & Analysis (10 minutes)

Let's compare all three approaches.

### 🎯 Exercise 1 — Bring in your Part A result

**Task:** if you ran Part A in this Colab session, its `test_predictions.csv` is picked up automatically. Otherwise set `FROM_SCRATCH_ACC` to the test accuracy you obtained in Part A (e.g. `0.93`).

**Expected output:** a three-row comparison table. Fine-tuned ResNet50 usually beats the from-scratch CNN by a few points; Helber et al. (2019) report 98.57% for a fine-tuned ResNet-50 on EuroSAT RGB.
1. **From-Scratch CNN** (Part A)
2. **Transfer Learning - Feature Extraction**
3. **Transfer Learning - Fine-Tuned**

In [ ]:
# Summary comparison
FROM_SCRATCH_ACC = None  # TODO: your Part A test accuracy, e.g. 0.93 (leave None to read test_predictions.csv)

if FROM_SCRATCH_ACC is None and Path('test_predictions.csv').exists():
    FROM_SCRATCH_ACC = pd.read_csv('test_predictions.csv')['correct'].mean()
    print(f"Read Part A result from test_predictions.csv: {FROM_SCRATCH_ACC:.3f}")
from_scratch_acc = FROM_SCRATCH_ACC if FROM_SCRATCH_ACC is not None else float('nan')

trainable_ft = sum(int(np.prod(w.shape)) for w in model_feature_extraction.trainable_weights)
df_comparison = pd.DataFrame({
    'Method': ['From-scratch CNN (Part A)', 'ResNet50 feature extraction', 'ResNet50 fine-tuned'],
    'Test accuracy (%)': [from_scratch_acc * 100, test_acc_fe * 100, test_acc_ft * 100],
    'Trainable params': ['see Part A', f'{head_params:,}', f'{trainable_ft:,}'],
    'Training time (min)': ['see Part A', f'{time_fe/60:.1f}', f'{time_ft/60:.1f}'],
})

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(df_comparison.round(2).to_string(index=False))
print("=" * 70)

### Visualize Comparison

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['From-Scratch\nCNN', 'ResNet50\nFeature Ext.', 'ResNet50\nFine-Tuned']
accuracies = [from_scratch_acc * 100, test_acc_fe * 100, test_acc_ft * 100]
colors = ['steelblue', 'orange', 'green']

bars = ax.bar(methods, accuracies, color=colors, edgecolor='black', linewidth=2, alpha=0.8)

ax.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('EuroSAT Classification: Model Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(max(0, np.nanmin(accuracies) - 10), 100)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.3,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add baseline reference
ax.axhline(y=90, color='red', linestyle='--', linewidth=2, alpha=0.5, label='90% threshold')
ax.legend()

plt.tight_layout()
plt.show()

print("\n✓ Transfer learning provides significant improvement!")

### Confusion Matrix (Fine-Tuned Model)

In [ ]:
# Generate predictions for confusion matrix
print("Generating predictions...")

y_true = []
y_pred = []

for images, labels in ds_test:
    predictions = model_feature_extraction.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax)

ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix - ResNet50 Fine-Tuned', fontsize=14, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("✓ Confusion matrix for fine-tuned model")

### Per-Class Performance

In [ ]:
# Classification report
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print("\n📊 Per-Class Metrics (Fine-Tuned Model):")
print("=" * 80)
print(report_df[:-3].round(3))  # Exclude averages
print("=" * 80)

# Identify best and worst
metrics_df = report_df[:-3]
best_class = metrics_df['f1-score'].idxmax()
worst_class = metrics_df['f1-score'].idxmin()

print(f"\n✨ Best: {best_class} (F1={metrics_df.loc[best_class, 'f1-score']:.3f})")
print(f"⚠️  Worst: {worst_class} (F1={metrics_df.loc[worst_class, 'f1-score']:.3f})")

---

# 🎉 Transfer Learning Lab Complete!

## Summary

You've successfully:

✅ **Loaded** pre-trained ResNet50 (25M parameters from ImageNet)  
✅ **Feature Extraction:** Trained a new classifier head on frozen ImageNet features  
✅ **Fine-Tuned:** Adapted the top of ResNet50 to satellite imagery  
✅ **Compared:** Measured the gain over the from-scratch CNN  
(For reference, Helber et al. 2019 report 98.57% on EuroSAT RGB with a fine-tuned ResNet-50.)  

---

## Key Insights

### Why Transfer Learning Works

1. **Pre-trained features are universal**
   - Edges, textures, patterns learned on ImageNet
   - Apply to satellite imagery without retraining

2. **Less data required**
   - ImageNet: 1.2M images
   - EuroSAT: 27K images
   - Transfer learning bridges the gap

3. **Faster convergence**
   - Start from good weights
   - 5-15 min vs 15-30 min from scratch

4. **Better accuracy**
   - +3-6% improvement
   - Critical for real-world applications

### When to Use Transfer Learning

✅ **Use transfer learning when:**
- Limited training data (<10K images)
- Similar task (image classification)
- Time/compute constrained
- Need best accuracy

❌ **Train from scratch when:**
- Very different domain (medical, satellite with many bands)
- Abundant data (>100K images)
- Specific architectural requirements
- Learning about CNNs (educational)

---

## Comparison Summary

| Metric | From-Scratch | Feature Extraction | Fine-Tuned |
|--------|--------------|-------------------|------------|
| **Accuracy** | your Part A result | see table above | see table above |
| **Training time** | see Part A | measured above | measured above |
| **Parameters trained** | ~2.2M | ~1M (head only) | head + last 20 layers |
| **Data Efficiency** | Needs more | Good | Best |
| **Overfitting Risk** | Higher | Low | Medium |

---

## Philippine Applications

**Transfer learning is ideal for:**

1. **Mangrove Mapping**
   - Limited labeled data
   - High accuracy needed
   - ResNet50 → Fine-tune on Palawan mangroves

2. **Rice Paddy Detection**
   - Seasonal patterns matter more than single-date texture
   - Combine with time series (Day 4) rather than single chips

3. **Informal Settlement Detection**
   - Needs very-high-resolution imagery (not 10 m Sentinel-2)
   - ResNet50 → Fine-tune on Metro Manila

4. **Disaster Damage Assessment**
   - Limited post-disaster data
   - Transfer from pre-trained → Quick deployment

---

## Next Steps

### Continue to Day 3: U-Net Segmentation
- Pixel-level classification
- Encoder-decoder architecture with skip connections
- Flood mapping with Sentinel-1

### Experiments to Try

**Easy:**
1. Unfreeze different numbers of layers (10, 30, 50)
2. Try different learning rates (1e-4, 1e-6)
3. Compare ResNet50 vs VGG16

**Medium:**
4. Use different pre-trained models (EfficientNet, MobileNet)
5. Multi-spectral adaptation: the 13-band EuroSAT_MS.zip (same Zenodo record) with a new first layer — ImageNet weights only cover RGB
6. Apply to Palawan dataset

**Advanced:**
7. Progressive unfreezing (unfreeze layers gradually)
8. Discriminative learning rates (different LR per layer)
9. Ensemble multiple fine-tuned models

---

## Save Models

In [ ]:
# Save fine-tuned model
model_feature_extraction.save('resnet50_eurosat_finetuned_final.keras')
print("✓ Fine-tuned model saved")

# Save training histories
import pickle
with open('transfer_learning_history.pkl', 'wb') as f:
    pickle.dump({
        'feature_extraction': history_fe.history,
        'fine_tuning': history_ft.history
    }, f)
print("✓ Training histories saved")

# Export results
results = pd.DataFrame({
    'Method': ['From-Scratch', 'Feature Extraction', 'Fine-Tuned'],
    'Test_Accuracy': [from_scratch_acc, test_acc_fe, test_acc_ft]
})
results.to_csv('transfer_learning_comparison.csv', index=False)
print("✓ Comparison results saved")

print("\n🎊 All done! Next: U-Net segmentation on Day 3.")

---

*EO AI/ML Training · Day 2 · Dimitrios Kasampalis (Anefelo) · [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)*